# pipe_demanda — top-down por categoria + cointegracion (todo en un notebook)

Version consolidada de `nat_exp/05_TopDown_completo.ipynb`, en un notebook
unico: preprocesamiento a nivel PRODUCTO (densificacion completa, estilo
`workflow`), modelo del TOTAL de una categoria (`nivel_agregado='cat3'` son
categorias comerciales especificas -- sopas, jabon liquido, ...), modelo del
SHARE de cada producto dentro de su categoria, y `tn = total x share
renormalizado`.

Se agregan dos cosas que `05_TopDown_completo` no tenia:

1. **Cointegracion** (portfolio management / pairs trading): por producto,
   test de Engle-Granger entre su `tn` y el `tn` de su categoria. Si estan
   cointegrados, el share es mean-reverting y se predice con un AR(1) en vez
   de LightGBM -- un cuarto enfoque, `topdown_cointegrado`, al lado de
   `topdown` (share via LightGBM), `bottomup`, `ma3` y `naive`.
2. **Cruce con productos magicos** (`nat_exp/residuos_2.ipynb`): tabla
   exploratoria comparando tasa de cointegracion y WAPE entre productos
   magicos y el resto -- para ver si hay un patron antes de construir nada
   sobre esa hipotesis.


In [ ]:
import json, os, shutil, subprocess, time
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb
import optuna
from statsmodels.tsa.stattools import coint

optuna.logging.set_verbosity(optuna.logging.WARNING)


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET   = resolver_bucket()
DIR_RAW  = BUCKET / "datasets"
DIR_FE   = BUCKET / "datasets_fe"     # de aca se lee productos_magicos.json (residuos_2)
RUTA_EXP = BUCKET / "exp_demanda"
RUTA_EXP.mkdir(parents=True, exist_ok=True)


def _leer_json_reintentando(path, intentos=5, espera=2):
    """El bucket es un mount GCS FUSE: a veces tira Input/output error transitorio."""
    ultimo_error = None
    for _ in range(intentos):
        try:
            with open(path, encoding="utf-8") as f:
                return json.load(f)
        except OSError as e:
            ultimo_error = e
            time.sleep(espera)
    raise ultimo_error

print(f"BUCKET : {BUCKET}")
print(f"crudos : {DIR_RAW}")
print(f"salida : {RUTA_EXP}")


def rango_meses(desde: int, hasta: int) -> list:
    a, b = (desde // 100) * 12 + desde % 100, (hasta // 100) * 12 + hasta % 100
    return [((m - 1) // 12) * 100 + ((m - 1) % 12) + 1 for m in range(a, b + 1)]


### Palancas (identicas a `05_TopDown_completo.ipynb`, mas cointegracion y productos magicos)


In [ ]:
PARAM = {
    'horizonte': 2,

    # ── NIVEL DEL AGREGADO ───────────────────────────────────────────────
    # 'cat3' es el nivel mas fino de tb_productos (categorias comerciales
    # especificas: sopas, jabon liquido, ...). 'cat2'/'cat1' agregan mas
    # grueso; 'brand' agrupa por marca; 'mercado' es un unico total.
    'nivel_agregado': 'cat3',

    # ── Particion temporal (mismos defaults que el resto de la sesion) ───
    'meses_train': rango_meses(201701, 201905),
    'meses_val':   [201907, 201908],
    'meses_test':  [201910],
    'reentrenar_con_val_para_test': True,

    # ── Densificacion ────────────────────────────────────────────────────
    'densificar': 'full',

    # ── Features ─────────────────────────────────────────────────────────
    'max_lags': 12,
    'meses_nuevo': 6,

    # ── Modelo del total del agregado ────────────────────────────────────
    'modelo_total': 'auto',

    # ── Optuna sobre el modelo del SHARE (topdown) y sobre bottomup ──────
    'n_trials': 40,
    'techo_arboles': 500,

    # ── Cointegracion (NUEVO) ─────────────────────────────────────────────
    # Engle-Granger entre tn del producto y tn de su categoria, SOLO con
    # meses de train y desde que el producto nacio (causal). p-value por
    # debajo de esto -> cointegrado -> se predice el share con reversion a
    # la media en vez de LightGBM.
    'cointegracion_alpha': 0.05,
    # Minimo de meses de historia (desde que nacio, hasta el corte de train)
    # para siquiera intentar el test -- con pocas observaciones el test de
    # Engle-Granger no es confiable, y un producto asi queda 'no cointegrado'
    # por seguridad (usa el share de LightGBM).
    'min_meses_cointegracion': 24,

    # ── Productos magicos (NUEVO) ─────────────────────────────────────────
    # El productos_magicos.json que escribe nat_exp/residuos_2.ipynb (en
    # datasets_fe). None = no se cruza nada.
    'archivo_productos_magicos': 'productos_magicos.json',

    # ── Entrega ──────────────────────────────────────────────────────────
    'periodo_objetivo': 202002,
    'semillas_ensemble': [102191],
    'clip_min': 0.0,
    'kaggle_competition': 'labo-iii-2026-rosario',
    'submit': False,
    'mensaje_submit': None,

    'semilla': 102191,
    'sufijo': '',
}

H = PARAM['horizonte']
NIVEL = PARAM['nivel_agregado']

EXPERIMENTO = (f"demanda_{NIVEL}_{PARAM['densificar']}_{PARAM['max_lags']}lags"
              f"_total-{PARAM['modelo_total']}"
              f"_val{PARAM['meses_val'][0]}-{PARAM['meses_val'][-1]}"
              f"_test{PARAM['meses_test'][0]}"
              + (f"_{PARAM['sufijo']}" if PARAM['sufijo'] else ""))
DIR_OUT = RUTA_EXP / EXPERIMENTO
DIR_OUT.mkdir(parents=True, exist_ok=True)

print(f"EXPERIMENTO : {EXPERIMENTO}")
print(f"carpeta     : {DIR_OUT.relative_to(BUCKET)}")
print(f"nivel del agregado: {NIVEL}   horizonte: {H}")


### Panel producto-mes + densificacion (identico a `05_TopDown_completo.ipynb`)


In [ ]:
sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
          .unique(subset=["product_id"]))

print(f"sell-in  : {sell.height:,} filas · {sell['product_id'].n_unique()} productos "
     f"· {sell['customer_id'].n_unique()} clientes")


def a_m(periodo):
    return (periodo // 100) * 12 + (periodo % 100)


def m_a_periodo(m):
    return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1


panel = (sell.group_by(["product_id", "periodo"])
             .agg(pl.col("tn").sum().alias("tn"),
                  pl.col("cust_request_tn").sum().alias("req_tn"),
                  pl.col("cust_request_qty").sum().alias("req_qty"),
                  pl.col("customer_id").n_unique().alias("n_clientes"),
                  pl.col("plan_precios_cuidados").max().alias("precios_cuidados"))
             .with_columns((a_m(pl.col("periodo"))).alias("m")))

M_MIN, M_MAX = panel["m"].min(), panel["m"].max()
print(f"panel    : {panel.height:,} filas · rango {m_a_periodo(M_MIN)} -> {m_a_periodo(M_MAX)}")

vida = panel.group_by("product_id").agg(
    pl.col("m").min().alias("m_nace"), pl.col("m").max().alias("m_muere"))

if PARAM['densificar'] == 'full':
    grilla = (prod.select("product_id")
                  .join(pl.DataFrame({"m": list(range(M_MIN, M_MAX + 1))}), how="cross"))
else:
    grilla = (vida.with_columns(
                    pl.int_ranges("m_nace", pl.col("m_muere") + 1).alias("m"))
                  .explode("m").select("product_id", "m"))

panel = (grilla.join(panel.drop("periodo"), on=["product_id", "m"], how="left")
               .with_columns(pl.col("tn").fill_null(0.0),
                             pl.col("req_tn").fill_null(0.0),
                             pl.col("req_qty").fill_null(0),
                             pl.col("n_clientes").fill_null(0),
                             pl.col("precios_cuidados").fill_null(0))
               .join(vida, on="product_id", how="left")
               .join(prod.select("product_id", "cat1", "cat2", "cat3", "brand", "sku_size"),
                     on="product_id", how="left")
               .with_columns(pl.col("m").map_elements(m_a_periodo, return_dtype=pl.Int64)
                               .alias("periodo")))

if NIVEL == 'mercado':
    panel = panel.with_columns(pl.lit("MERCADO").alias("grupo"))
else:
    panel = panel.with_columns(pl.col(NIVEL).cast(pl.Utf8).fill_null("NA").alias("grupo"))

panel = panel.sort(["product_id", "m"])

print(f"densificado ({PARAM['densificar']}): {panel.height:,} filas · "
     f"{panel['product_id'].n_unique()} productos · {panel['grupo'].n_unique()} grupos")
print(f"ceros de tn: {(panel['tn'] == 0).sum():,} ({100*(panel['tn'] == 0).sum()/panel.height:.0f}%)")


### Total del agregado + panel del producto (share, lags, fase de ciclo de vida) — identico


In [ ]:
L = PARAM['max_lags']

tot = (panel.group_by(["grupo", "m"])
            .agg(pl.col("tn").sum().alias("tn_grupo"),
                 pl.len().alias("n_prod_grupo"))
            .sort(["grupo", "m"]))

tot = tot.with_columns(
    *[pl.col("tn_grupo").shift(k).over("grupo").alias(f"tn_grupo_lag{k}")
      for k in range(1, L + 1)],
    *[pl.col("tn_grupo").rolling_mean(w).over("grupo").alias(f"tn_grupo_ma{w}")
      for w in (3, 6, 12)],
    (((pl.col("m") - 1) % 12) + 1).alias("mes_del_anio_grupo"),
    pl.col("tn_grupo").shift(-H).over("grupo").alias("y_tn_grupo"),
)

df = panel.join(tot.select("grupo", "m", "tn_grupo", "n_prod_grupo"),
                on=["grupo", "m"], how="left")

df = df.with_columns(
    pl.when(pl.col("tn_grupo").abs() > 1e-9)
      .then(pl.col("tn") / pl.col("tn_grupo"))
      .otherwise(0.0).alias("share"))

df = df.sort(["product_id", "m"]).with_columns(
    *[pl.col("tn").shift(k).over("product_id").alias(f"tn_lag{k}") for k in range(1, L + 1)],
    *[pl.col("tn").rolling_mean(w).over("product_id").alias(f"tn_ma{w}") for w in (3, 6, 12)],
    *[pl.col("share").shift(k).over("product_id").alias(f"share_lag{k}") for k in range(1, L + 1)],
    *[pl.col("share").rolling_mean(w).over("product_id").alias(f"share_ma{w}") for w in (3, 6, 12)],
    pl.col("req_tn").rolling_mean(3).over("product_id").alias("req_tn_ma3"),
    pl.col("n_clientes").rolling_mean(3).over("product_id").alias("n_clientes_ma3"),
    pl.col("tn").cum_max().over("product_id").alias("tn_pico_hasta_aca"),
    pl.col("tn").cum_sum().over("product_id").alias("tn_acum"),
    pl.when(pl.col("m") >= pl.col("m_nace"))
      .then(pl.col("m") - pl.col("m_nace"))
      .otherwise(-1).alias("edad"),
    ((pl.col("periodo") % 100)).alias("mes_del_anio"),
)

df = df.with_columns(
    (pl.col("share") - pl.col("share_ma3")).alias("share_desvio_vs_ma3"),
    (pl.col("share") - pl.col("share_lag1")).alias("share_mom"),
    (pl.col("tn") - pl.col("tn_ma3")).alias("tn_desvio_vs_ma3"),
    pl.when(pl.col("tn_pico_hasta_aca") > 1e-9)
      .then(pl.col("tn") / pl.col("tn_pico_hasta_aca"))
      .otherwise(0.0).alias("frac_del_pico"),
    (pl.col("edad").is_between(0, PARAM['meses_nuevo'])).cast(pl.Int8).alias("es_nuevo"),
    (pl.col("tn") == 0).cast(pl.Int8).alias("mes_sin_venta"),
)

df = df.with_columns(
    pl.when(pl.col("edad") < 0).then(pl.lit("sin_lanzar"))
      .when(pl.col("edad") <= PARAM['meses_nuevo']).then(pl.lit("rampa"))
      .when(pl.col("frac_del_pico") >= 0.85).then(pl.lit("crecimiento"))
      .when(pl.col("frac_del_pico") >= 0.50).then(pl.lit("meseta"))
      .when(pl.col("frac_del_pico") > 0.0).then(pl.lit("declive"))
      .otherwise(pl.lit("dormido")).alias("fase"))

nacimientos = (df.filter(pl.col("m") == pl.col("m_nace"))
                 .group_by(["grupo", "m"]).agg(pl.len().alias("entradas")))
ventana = (tot.select("grupo", "m")
              .join(nacimientos, on=["grupo", "m"], how="left")
              .with_columns(pl.col("entradas").fill_null(0))
              .sort(["grupo", "m"])
              .with_columns(pl.col("entradas").rolling_sum(6).over("grupo").alias("entradas_6m")))

df = (df.join(ventana.select("grupo", "m", "entradas_6m"), on=["grupo", "m"], how="left")
        .join(tot.select(["grupo", "m"] + [c for c in tot.columns
                                           if c.startswith("tn_grupo_")]),
              on=["grupo", "m"], how="left"))

df = df.sort(["product_id", "m"]).with_columns(
    pl.col("share").shift(-H).over("product_id").alias("y_share"),
    pl.col("tn").shift(-H).over("product_id").alias("y_tn"),
    (pl.col("m") + H).map_elements(m_a_periodo, return_dtype=pl.Int64)
      .alias("periodo_objetivo"),
)
df = df.join(tot.select("grupo", "m", "y_tn_grupo"), on=["grupo", "m"], how="left")

FEATURES = [c for c in df.columns if c not in {
    "product_id", "periodo", "m", "grupo", "m_nace", "m_muere",
    "y_share", "y_tn", "y_tn_grupo", "periodo_objetivo",
    "cat1", "cat2", "cat3", "brand",
}]
CAT_FEATURES = ["cat1", "cat2", "cat3", "brand", "fase"]
FEATURES = [c for c in FEATURES if c not in CAT_FEATURES] + CAT_FEATURES

df = df.with_columns([pl.col(c).cast(pl.Utf8).fill_null("NA").cast(pl.Categorical)
                      for c in CAT_FEATURES])

print(f"panel de features: {df.height:,} filas x {len(FEATURES)} features")
print(f"categoricas: {CAT_FEATURES}")
print(df.group_by("fase").len().sort("len", descending=True))


### Split y control de leakage (identico)


In [ ]:
MESES_TRAIN = sorted(PARAM['meses_train'])
MESES_VAL   = sorted(PARAM['meses_val'])
MESES_TEST  = sorted(PARAM['meses_test'])

sup = df.filter(pl.col("y_share").is_not_null() & pl.col("y_tn").is_not_null())
periodos_sup = sorted(sup["periodo"].unique().to_list())

MESES_TRAIN = [m for m in MESES_TRAIN if m in periodos_sup]
MESES_VAL   = [m for m in MESES_VAL   if m in periodos_sup]
MESES_TEST  = [m for m in MESES_TEST  if m in periodos_sup]

MESES_INFER = sorted(df.filter(pl.col("y_tn").is_null())["periodo"].unique().to_list())[-H:]
infer = df.filter(pl.col("periodo").is_in(MESES_INFER))

errores = []


def chk(ok, msg):
    print(f"  [{'ok   ' if ok else 'ERROR'}] {msg}")
    if not ok:
        errores.append(msg)


print("CONTROL DE LEAKAGE")
print("=" * 72)

for a, b, na, nb in ((MESES_TRAIN, MESES_VAL, "train", "val"),
                    (MESES_VAL, MESES_TEST, "val", "test")):
    gap = a_m(min(b)) - a_m(max(a))
    chk(gap >= H, f"gap {na}({max(a)}) -> {nb}({min(b)}) = {gap} mes(es) >= horizonte {H}")

if PARAM['reentrenar_con_val_para_test']:
    gap_tv = a_m(min(MESES_TEST)) - a_m(max(MESES_TRAIN + MESES_VAL))
    chk(gap_tv >= H, f"gap (train+val) -> test = {gap_tv} >= {H}")

chk(not (set(MESES_TRAIN) & set(MESES_VAL)), "train y val son disjuntos")
chk(not (set(MESES_VAL) & set(MESES_TEST)), "val y test son disjuntos")
chk(max(MESES_TRAIN) < min(MESES_VAL) < max(MESES_VAL) < min(MESES_TEST),
    "orden cronologico train < val < test")
chk(not (set(MESES_INFER) & set(MESES_TRAIN + MESES_VAL + MESES_TEST)),
    f"los meses de inferencia {MESES_INFER} no se usan para entrenar/validar/testear")

prohibidas = {"y_share", "y_tn", "y_tn_grupo", "periodo_objetivo"}
chk(not (set(FEATURES) & prohibidas), "ningun target esta entre las features")

y_chk = sup["y_tn"].to_numpy().astype(np.float64)
sospechosas = []
for c in FEATURES:
    if c in CAT_FEATURES:
        continue
    x = sup[c].to_numpy().astype(np.float64)
    ok = np.isfinite(x) & np.isfinite(y_chk)
    if ok.sum() < 100 or x[ok].std() == 0:
        continue
    r = float(np.corrcoef(x[ok], y_chk[ok])[0, 1])
    if abs(r) > 0.999:
        sospechosas.append((c, round(r, 5)))
chk(not sospechosas, f"ninguna feature correlaciona >0.999 con y_tn  {sospechosas}")

print("=" * 72)
print(f"TRAIN {len(MESES_TRAIN)} meses: {MESES_TRAIN[0]}..{MESES_TRAIN[-1]}"
     f"   ({sup.filter(pl.col('periodo').is_in(MESES_TRAIN)).height:,} filas)")
print(f"VAL   {len(MESES_VAL)} meses: {MESES_VAL}"
     f"   ({sup.filter(pl.col('periodo').is_in(MESES_VAL)).height:,} filas)")
print(f"TEST  {len(MESES_TEST)} meses: {MESES_TEST}"
     f"   ({sup.filter(pl.col('periodo').is_in(MESES_TEST)).height:,} filas)")
print(f"INFER {len(MESES_INFER)} meses: {MESES_INFER}   ({infer.height:,} filas)"
     f"  -> objetivo {sorted(infer['periodo_objetivo'].unique().to_list())}")

if errores:
    raise RuntimeError(f"Control de leakage FALLIDO: {errores}")
print("\nControl superado.")


### WAPE y reconstruccion (identico)


In [ ]:
def wape(y_real, y_pred, product_ids=None, por_producto=True) -> float:
    """WAPE en toneladas. Identico al del resto de la sesion."""
    yr = np.asarray(y_real, dtype=np.float64)
    yp = np.maximum(np.asarray(y_pred, dtype=np.float64), 0.0)
    if por_producto and product_ids is not None:
        _, inv = np.unique(np.asarray(product_ids), return_inverse=True)
        yr = np.bincount(inv, weights=yr)
        yp = np.bincount(inv, weights=yp)
    den = np.abs(yr).sum()
    return float("nan") if den == 0 else float(np.abs(yr - yp).sum() / den)


def reconstruir(bloque: pl.DataFrame, share_pred, total_pred) -> np.ndarray:
    """tn_predicho = total_del_grupo_predicho x share_renormalizado."""
    aux = bloque.select("grupo", "m").with_columns(
        pl.Series("s", np.maximum(np.asarray(share_pred, dtype=np.float64), 0.0)),
        pl.Series("tot", np.asarray(total_pred, dtype=np.float64)))
    aux = aux.with_columns(pl.col("s").sum().over(["grupo", "m"]).alias("s_suma"))
    aux = aux.with_columns(
        pl.when(pl.col("s_suma") > 1e-12)
          .then(pl.col("s") / pl.col("s_suma"))
          .otherwise(0.0).alias("s_norm"))
    return (aux["s_norm"] * aux["tot"]).to_numpy()


def wape_de(bloque: pl.DataFrame, pred_tn) -> float:
    return wape(bloque["y_tn"].to_numpy(), pred_tn, bloque["product_id"].to_numpy())


def bloques(meses):
    return sup.filter(pl.col("periodo").is_in(meses))


tr, va, te = bloques(MESES_TRAIN), bloques(MESES_VAL), bloques(MESES_TEST)
print(f"train {tr.height:,} · val {va.height:,} · test {te.height:,} filas")


### Modelo del total del agregado: `ma3` vs LightGBM, por validacion (identico)


In [ ]:
FEAT_TOT = [c for c in tot.columns if c not in {"grupo", "m", "y_tn_grupo"}]
tot_all = tot.with_columns(
    pl.col("m").map_elements(m_a_periodo, return_dtype=pl.Int64).alias("periodo"))
tot_sup = tot_all.filter(pl.col("y_tn_grupo").is_not_null())


def tot_bloque(meses):
    return tot_sup.filter(pl.col("periodo").is_in(meses))


PARAMS_TOT = {'objective': 'regression_l1', 'metric': 'mae', 'verbosity': -1,
             'n_estimators': 400, 'learning_rate': 0.05, 'num_leaves': 31,
             'min_child_samples': 20, 'seed': PARAM['semilla'], 'n_jobs': -1,
             'deterministic': True, 'force_row_wise': True}


def entrenar_total(meses):
    b = tot_bloque(meses).to_pandas()
    m = lgb.LGBMRegressor(**PARAMS_TOT)
    m.fit(b[FEAT_TOT], b["y_tn_grupo"])
    return m


def predecir_total(modelo_lgbm, bloque_tot, metodo):
    if metodo == 'ma3':
        return bloque_tot["tn_grupo_ma3"].fill_null(0.0).to_numpy()
    return modelo_lgbm.predict(bloque_tot.to_pandas()[FEAT_TOT])


_m_tot_tr = entrenar_total(MESES_TRAIN)
_va_tot = tot_bloque(MESES_VAL)
wape_tot = {}
for metodo in ('ma3', 'lgbm'):
    p = np.maximum(predecir_total(_m_tot_tr, _va_tot, metodo), 0.0)
    wape_tot[metodo] = wape(_va_tot["y_tn_grupo"].to_numpy(), p, _va_tot["grupo"].to_numpy())
    print(f"  total {metodo:5s} -> WAPE del agregado en val = {wape_tot[metodo]:.5f}")

METODO_TOTAL = (PARAM['modelo_total'] if PARAM['modelo_total'] != 'auto'
               else min(wape_tot, key=wape_tot.get))
print(f"\nmodelo del total elegido: {METODO_TOTAL}"
     + ("  (por validacion)" if PARAM['modelo_total'] == 'auto' else "  (forzado en PARAM)"))


def totales_para(bloque, meses_fit):
    mt = entrenar_total(meses_fit) if METODO_TOTAL == 'lgbm' else None
    tb = tot_all.filter(pl.col("periodo").is_in(sorted(bloque["periodo"].unique().to_list())))
    pred = np.maximum(predecir_total(mt, tb, METODO_TOTAL), 0.0)
    mapa = tb.select("grupo", "m").with_columns(pl.Series("tot_pred", pred))
    return (bloque.select("grupo", "m").join(mapa, on=["grupo", "m"], how="left")
                  ["tot_pred"].fill_null(0.0).to_numpy())


### Cointegracion Engle-Granger (NUEVO)

Por producto: test de Engle-Granger entre su `tn` y el `tn_grupo` de su
categoria (`statsmodels.tsa.stattools.coint`), usando SOLO meses de
`MESES_TRAIN` (causal) y SOLO desde `m_nace` del producto (los ceros de
antes de que naciera no son no-estacionariedad real, son "todavia no
existia"). Si `pvalue < PARAM['cointegracion_alpha']`, el producto es
cointegrado con su categoria -> su share es mean-reverting, y se ajusta un
AR(1) sobre la serie de SHARE (no sobre el spread de niveles: mas directo,
misma idea de reversion) para poder predecir `share_pred = mu + phi**H *
(share_t - mu)`. Productos con menos de `min_meses_cointegracion` de
historia quedan `cointegrado=False` por seguridad (el test no es confiable
con pocas observaciones).


In [ ]:
t0 = time.time()
M_CORTE_TRAIN = max(MESES_TRAIN)


def _ar1(x):
    """Ajusta x[t] = mu + phi*(x[t-1] - mu) por minimos cuadrados. Devuelve (phi, mu)."""
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if len(x) < 8:
        return 0.0, float(np.mean(x)) if len(x) else 0.0
    y, x_lag = x[1:], x[:-1]
    mu0 = float(np.mean(x))
    xc = x_lag - mu0
    denom = float(np.sum(xc ** 2))
    phi = float(np.sum((y - mu0) * xc) / denom) if denom > 1e-12 else 0.0
    phi = max(min(phi, 0.98), -0.98)   # evita explosion si el ajuste sale mal
    return phi, mu0


filas_coint = []
_prod_ids = df["product_id"].unique().to_list()
_train_m = set(a_m(p) for p in MESES_TRAIN)

for pid in _prod_ids:
    serie = (df.filter((pl.col("product_id") == pid) & pl.col("m").is_in(_train_m))
               .sort("m"))
    serie = serie.filter(pl.col("m") >= pl.col("m_nace"))
    n_obs = serie.height
    grupo_pid = serie["grupo"][0] if n_obs else None
    cointegrado, pvalue, phi, mu = False, None, 0.0, 0.0

    if n_obs >= PARAM['min_meses_cointegracion']:
        tn_p = serie["tn"].to_numpy().astype(np.float64)
        tn_g = serie["tn_grupo"].to_numpy().astype(np.float64)
        if np.std(tn_p) > 1e-9 and np.std(tn_g) > 1e-9:
            try:
                _, pvalue, _ = coint(tn_p, tn_g)
                cointegrado = bool(pvalue < PARAM['cointegracion_alpha'])
            except Exception:
                cointegrado, pvalue = False, None

    if cointegrado:
        phi, mu = _ar1(serie["share"].to_numpy())

    filas_coint.append({'product_id': pid, 'grupo': grupo_pid, 'n_obs': n_obs,
                        'pvalue': pvalue, 'cointegrado': cointegrado,
                        'phi': round(phi, 4), 'mu': round(mu, 6)})

cointegracion = pl.DataFrame(filas_coint)
n_coint = int(cointegracion["cointegrado"].sum())
print(f"productos cointegrados con su categoria: {n_coint} de {cointegracion.height} "
     f"({100*n_coint/cointegracion.height:.0f}%)")
print(cointegracion.sort("pvalue").head(10))

cointegracion.write_parquet(DIR_OUT / "cointegracion.parquet")
print(f"Guardado: {DIR_OUT / 'cointegracion.parquet'}")
print(f"[{time.time()-t0:.0f}s]")


### Enfoque `topdown_cointegrado` (NUEVO)

Por producto: si esta cointegrado, el share se predice con el AR(1)
(reversion a la media); si no, se usa el share que predice LightGBM
(`topdown`, mas abajo). Se combina DESPUES de tener las dos predicciones de
share -- esta funcion solo necesita la tabla de cointegracion y el share
"crudo" del bloque (para poder aplicar la formula de reversion).


In [ ]:
_MAPA_COINT = {r['product_id']: r for r in cointegracion.to_dicts()}


def share_cointegrado(bloque: pl.DataFrame, share_lgbm_pred) -> np.ndarray:
    """share_pred: reversion a la media donde cointegrado, share de LightGBM
    donde no. bloque trae product_id y share (el share EN t, para poder
    revertir hacia mu) alineado fila a fila con share_lgbm_pred."""
    pids = bloque["product_id"].to_list()
    share_t = bloque["share"].to_numpy().astype(np.float64)
    out = np.asarray(share_lgbm_pred, dtype=np.float64).copy()
    for i, pid in enumerate(pids):
        info = _MAPA_COINT.get(pid)
        if info is not None and info['cointegrado']:
            out[i] = info['mu'] + (info['phi'] ** H) * (share_t[i] - info['mu'])
    return np.maximum(out, 0.0)


### Optuna sobre el share (`topdown`) y sobre `bottomup` (identico)


In [ ]:
def espacio(trial):
    return {
        'objective': 'regression_l1', 'metric': 'mae', 'verbosity': -1,
        'seed': PARAM['semilla'], 'n_jobs': -1,
        'deterministic': True, 'force_row_wise': True,
        'num_leaves':        trial.suggest_int('num_leaves', 15, 255),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'learning_rate':     trial.suggest_float('learning_rate', 5e-3, 0.3, log=True),
        'n_estimators':      trial.suggest_int('n_estimators', 100, PARAM['techo_arboles']),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 200),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq':    1,
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }


def fit_lgbm(params, meses, target):
    b = bloques(meses).to_pandas()
    m = lgb.LGBMRegressor(**params)
    m.fit(b[FEATURES], b[target], categorical_feature=CAT_FEATURES)
    return m


TOT_VAL = totales_para(va, MESES_TRAIN)


def obj_topdown(trial):
    m = fit_lgbm(espacio(trial), MESES_TRAIN, "y_share")
    s = m.predict(va.to_pandas()[FEATURES])
    return wape_de(va, reconstruir(va, s, TOT_VAL))


def obj_bottomup(trial):
    m = fit_lgbm(espacio(trial), MESES_TRAIN, "y_tn")
    return wape_de(va, m.predict(va.to_pandas()[FEATURES]))


estudios = {}
for nombre, objetivo in (("topdown", obj_topdown), ("bottomup", obj_bottomup)):
    t0 = time.time()
    st = optuna.create_study(
        direction="minimize", study_name=f"{EXPERIMENTO}__{nombre}",
        sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
        storage=f"sqlite:///{Path.home() / f'optuna_demanda_{nombre}.db'}",
        load_if_exists=True)
    st.optimize(objetivo, n_trials=PARAM['n_trials'])
    estudios[nombre] = st
    print(f"{nombre:9s} {len(st.trials)} trials · mejor WAPE val = {st.best_value:.5f}"
         f"  ({time.time()-t0:.0f}s)")


### Evaluacion val/test: `topdown`, `topdown_cointegrado` (NUEVO), `bottomup`, `ma3`, `naive`


In [ ]:
MESES_FIT_TEST = (MESES_TRAIN + MESES_VAL) if PARAM['reentrenar_con_val_para_test'] else MESES_TRAIN

P_TD = {**estudios['topdown'].best_params, 'objective': 'regression_l1', 'metric': 'mae',
       'verbosity': -1, 'seed': PARAM['semilla'], 'n_jobs': -1, 'subsample_freq': 1,
       'deterministic': True, 'force_row_wise': True}
P_BU = {**estudios['bottomup'].best_params, 'objective': 'regression_l1', 'metric': 'mae',
       'verbosity': -1, 'seed': PARAM['semilla'], 'n_jobs': -1, 'subsample_freq': 1,
       'deterministic': True, 'force_row_wise': True}


def evaluar(bloque, meses_fit):
    """Devuelve {enfoque: pred_tn} para un bloque de evaluacion."""
    tot_pred = totales_para(bloque, meses_fit)
    bp = bloque.to_pandas()

    m_sh = fit_lgbm(P_TD, meses_fit, "y_share")
    m_bu = fit_lgbm(P_BU, meses_fit, "y_tn")

    share_lgbm = m_sh.predict(bp[FEATURES])
    share_coint = share_cointegrado(bloque, share_lgbm)

    return {
        "topdown":            reconstruir(bloque, share_lgbm, tot_pred),
        "topdown_cointegrado": reconstruir(bloque, share_coint, tot_pred),
        "bottomup":           m_bu.predict(bp[FEATURES]),
        "naive":              bloque["tn"].to_numpy(),
        "ma3":                bloque["tn_ma3"].fill_null(0.0).to_numpy(),
    }, (m_sh, m_bu)


pred_val, _ = evaluar(va, MESES_TRAIN)
pred_test, (mod_sh_test, mod_bu_test) = evaluar(te, MESES_FIT_TEST)

ENFOQUES = ("topdown", "topdown_cointegrado", "bottomup", "ma3", "naive")
print(f"{'enfoque':22s} {'WAPE val':>10s} {'WAPE test':>10s}")
print("-" * 44)
METRICAS = {}
for enfoque in ENFOQUES:
    wv = wape_de(va, pred_val[enfoque])
    wt = wape_de(te, pred_test[enfoque])
    METRICAS[enfoque] = {"val": wv, "test": wt}
    print(f"{enfoque:22s} {wv:10.5f} {wt:10.5f}")

_td, _tdc, _bu = METRICAS['topdown']['test'], METRICAS['topdown_cointegrado']['test'], METRICAS['bottomup']['test']
_na = METRICAS['naive']['test']
print(f"\ntopdown_cointegrado vs topdown en test : {100*(_td-_tdc)/_td:+.1f}%"
     f"   ({'ayuda la cointegracion' if _tdc < _td else 'topdown puro ya era mejor'})")
print(f"top-down vs bottom-up en test           : {100*(_bu-_td)/_bu:+.1f}%")
print(f"mejor enfoque vs naive en test           : "
     f"{100*(_na-min(METRICAS[e]['test'] for e in ENFOQUES if e!='naive'))/_na:+.1f}%")

_brecha = METRICAS['topdown']['test'] - METRICAS['topdown']['val']
print(f"\nbrecha test - val del topdown: {_brecha:+.5f}"
     + ("   <- ojo, sobreajuste a validacion" if _brecha > 0.02 else ""))


### Donde gana cada uno (por fase y por edad) — identico, mas cointegracion


In [ ]:
det = te.select("product_id", "periodo", "fase", "edad", "y_tn").with_columns(
    pl.Series("td", pred_test["topdown"]),
    pl.Series("tdc", pred_test["topdown_cointegrado"]),
    pl.Series("bu", pred_test["bottomup"]))

filas = []
for v in det["fase"].unique().to_list():
    b = det.filter(pl.col("fase") == v)
    if b.height < 20:
        continue
    filas.append({"corte": f"fase={v}", "n": b.height,
                  "tn_real": round(float(b["y_tn"].sum()), 1),
                  "wape_topdown": round(wape(b["y_tn"], b["td"], b["product_id"]), 4),
                  "wape_topdown_coint": round(wape(b["y_tn"], b["tdc"], b["product_id"]), 4),
                  "wape_bottomup": round(wape(b["y_tn"], b["bu"], b["product_id"]), 4)})

for lo, hi, nombre in ((0, 6, "0-6m"), (7, 12, "7-12m"), (13, 24, "13-24m"), (25, 999, "25m+")):
    b = det.filter(pl.col("edad").is_between(lo, hi))
    if b.height < 20:
        continue
    filas.append({"corte": f"edad={nombre}", "n": b.height,
                  "tn_real": round(float(b["y_tn"].sum()), 1),
                  "wape_topdown": round(wape(b["y_tn"], b["td"], b["product_id"]), 4),
                  "wape_topdown_coint": round(wape(b["y_tn"], b["tdc"], b["product_id"]), 4),
                  "wape_bottomup": round(wape(b["y_tn"], b["bu"], b["product_id"]), 4)})

if not filas:
    print("Ningun corte (fase/edad) llega a las 20 filas minimas en este test -- "
         "seguramente hay pocos meses/productos de test. Se salta esta tabla.")
else:
    donde = (pl.DataFrame(filas)
               .with_columns((pl.col("wape_bottomup") - pl.col("wape_topdown")).round(4)
                             .alias("ventaja_topdown"))
               .sort("ventaja_topdown", descending=True))
    print(donde)
    donde.write_csv(DIR_OUT / "donde_gana_cada_uno.csv")


### Cruce con productos magicos (NUEVO, exploratorio)

`productos_magicos.json` lo escribe `nat_exp/residuos_2.ipynb` a nivel
producto (sin cliente, igual que este notebook), asi que el `product_id`
coincide directo. Es solo una tabla comparativa -- no se construye ninguna
regla nueva sobre esto todavia, primero hay que ver si el patron aparece.


In [ ]:
if PARAM['archivo_productos_magicos']:
    path_mag = DIR_FE / PARAM['archivo_productos_magicos']
    if not path_mag.exists():
        print(f"no encontre {path_mag} -- corre nat_exp/residuos_2.ipynb primero. "
             f"Se salta el cruce.")
    else:
        _meta_mag = _leer_json_reintentando(path_mag)
        _magicos = set(_meta_mag["product_ids"])
        print(f"productos magicos leidos de {path_mag.name}: {len(_magicos)} "
             f"(esquema={_meta_mag.get('esquema')} vs baseline={_meta_mag.get('baseline')})")

        cruce = (det.with_columns(pl.col("product_id").is_in(_magicos).alias("es_magico"))
                    .join(cointegracion.select("product_id", "cointegrado"),
                          on="product_id", how="left"))

        filas_cruce = []
        for es_mag in (True, False):
            b = cruce.filter(pl.col("es_magico") == es_mag)
            b_coint = cointegracion.filter(pl.col("product_id").is_in(
                set(cruce.filter(pl.col("es_magico") == es_mag)["product_id"].to_list())))
            if b.height < 5:
                continue
            filas_cruce.append({
                'grupo': 'magicos' if es_mag else 'resto',
                'n_productos': b["product_id"].n_unique(),
                'tasa_cointegracion': round(float(b_coint["cointegrado"].mean()), 3)
                                      if b_coint.height else None,
                'wape_topdown': round(wape(b["y_tn"], b["td"], b["product_id"]), 4),
                'wape_topdown_coint': round(wape(b["y_tn"], b["tdc"], b["product_id"]), 4),
                'wape_bottomup': round(wape(b["y_tn"], b["bu"], b["product_id"]), 4),
            })
        cruce_tabla = pl.DataFrame(filas_cruce)
        print(cruce_tabla)
        cruce_tabla.write_csv(DIR_OUT / "cruce_productos_magicos.csv")
        print(f"\nGuardado: {DIR_OUT / 'cruce_productos_magicos.csv'}")
        print("\nSolo exploratorio: si 'tasa_cointegracion' o los WAPE difieren mucho entre "
             "magicos y resto, hay un patron para investigar -- todavia no se arma ninguna "
             "regla sobre esto.")
else:
    print("PARAM['archivo_productos_magicos'] = None -> no se cruza nada.")


### Reentreno final y submit -- tres enfoques

Se piden los tres submits por separado (no se elige un "ganador" unico):
`topdown` (share via LightGBM), `topdown_cointegrado` (share via reversion a
la media donde el producto cointegra con su categoria) y `bottomup`
(LightGBM directo al nivel producto). `ma3`/`naive` quedan solo como
referencia en las metricas, no se suben.


In [ ]:
MESES_TODOS = sorted(set(periodos_sup))
print(f"reentrenando con {len(MESES_TODOS)} meses: {MESES_TODOS[0]}..{MESES_TODOS[-1]}")

tot_infer = totales_para(infer, MESES_TODOS)
ip = infer.to_pandas()

modelos_sh, modelos_bu = [], []
for sem in PARAM['semillas_ensemble']:
    modelos_sh.append(fit_lgbm({**P_TD, 'seed': sem}, MESES_TODOS, "y_share"))
    modelos_bu.append(fit_lgbm({**P_BU, 'seed': sem}, MESES_TODOS, "y_tn"))
    print(f"  semilla {sem} entrenada")

sh_pred = np.mean([m.predict(ip[FEATURES]) for m in modelos_sh], axis=0)
bu_pred = np.mean([m.predict(ip[FEATURES]) for m in modelos_bu], axis=0)
sh_coint_pred = share_cointegrado(infer, sh_pred)

pred_infer = infer.select("product_id", "periodo", "periodo_objetivo", "grupo").with_columns(
    pl.Series("tn_topdown",
              np.maximum(reconstruir(infer, sh_pred, tot_infer), PARAM['clip_min'])),
    pl.Series("tn_topdown_cointegrado",
              np.maximum(reconstruir(infer, sh_coint_pred, tot_infer), PARAM['clip_min'])),
    pl.Series("tn_bottomup", np.maximum(bu_pred, PARAM['clip_min'])))
pred_infer.write_parquet(DIR_OUT / "predicciones_inferencia.parquet")

print(f"\npredicciones: {pred_infer.height:,} filas")
print(pred_infer.group_by("periodo", "periodo_objetivo").len().sort("periodo"))

OBJ = PARAM['periodo_objetivo']
obj = pred_infer.filter(pl.col("periodo_objetivo") == OBJ)
if obj.is_empty():
    raise RuntimeError(f"No hay predicciones para {OBJ}. Disponibles: "
                       f"{sorted(pred_infer['periodo_objetivo'].unique().to_list())}")

oficiales = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt", separator="\t")

ENFOQUES_SUBMIT = ("topdown", "topdown_cointegrado", "bottomup")
submits = {}
for enf in ENFOQUES_SUBMIT:
    por_producto = obj.group_by("product_id").agg(pl.col(f"tn_{enf}").sum().alias("tn"))
    sub = (oficiales.select("product_id")
                    .join(por_producto, on="product_id", how="left"))
    sin_pred = int(sub["tn"].null_count())
    sub = sub.with_columns(pl.col("tn").fill_null(0.0)).sort("product_id")
    submits[enf] = {"df": sub, "sin_pred": sin_pred}
    print(f"{enf:22s} sin prediccion: {sin_pred:4d}/{oficiales.height}  "
         f"tn total: {sub['tn'].sum():,.1f}")
    if sin_pred > oficiales.height * 0.05:
        print(f"   ATENCION ({enf}): mas del 5% de la lista sin prediccion.")


In [ ]:
paths_submit = {}
for enf in ENFOQUES_SUBMIT:
    path_submit = DIR_OUT / f"submission_{OBJ}_{enf}.csv"
    submits[enf]["df"].write_csv(path_submit)
    paths_submit[enf] = path_submit
    print(f"Guardado: {path_submit}")
shutil.copy(paths_submit["topdown"], RUTA_EXP / "submission_ultima.csv")


def kaggle_cli(args):
    try:
        r = subprocess.run(["kaggle"] + args, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout or "") + (r.stderr or "")
    except FileNotFoundError:
        return False, ("La CLI de kaggle no esta instalada.  pip install kaggle\n"
                       "Los CSV ya quedaron generados; se pueden subir a mano.")
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


In [ ]:
if not PARAM['submit']:
    print("PARAM['submit'] = False -> no se sube nada. Los 3 CSV ya estan generados.")
else:
    kdst = Path.home() / ".kaggle" / "kaggle.json"
    kdst.parent.mkdir(parents=True, exist_ok=True)
    if not kdst.exists():
        for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
            if cand.exists():
                shutil.copy(cand, kdst); kdst.chmod(0o600)
                print(f"Kaggle auth copiada de {cand}")
                break
    if not kdst.exists():
        print("Sin credenciales de Kaggle: no se sube nada. Los CSV ya estan generados.")
    else:
        kdst.chmod(0o600)
        for enf in ENFOQUES_SUBMIT:
            msg = (PARAM['mensaje_submit'] or
                  f"{EXPERIMENTO} | {enf} | wape_test={METRICAS[enf]['test']:.5f}")
            ok, salida = kaggle_cli(["competitions", "submit",
                                    "-c", PARAM['kaggle_competition'],
                                    "-f", str(paths_submit[enf]), "-m", msg])
            print(f"[{enf}] mensaje: {msg}\n{salida}")
            print(f"[{enf}] " + ("Submit enviado." if ok else "NO se pudo subir; el CSV esta en disco."))


In [ ]:
resultado = {
    'experimento': EXPERIMENTO,
    'metodologia': 'top-down (total del agregado x share renormalizado) + cointegracion Engle-Granger',
    'nivel_agregado': NIVEL,
    'densificar': PARAM['densificar'],
    'horizonte': H,
    'max_lags': PARAM['max_lags'],
    'meses_train': MESES_TRAIN, 'meses_val': MESES_VAL, 'meses_test': MESES_TEST,
    'meses_inferencia': MESES_INFER, 'periodo_objetivo': OBJ,
    'modelo_total': METODO_TOTAL,
    'wape_total_agregado_val': wape_tot,
    'metricas': METRICAS,
    'enfoques_entregados': list(ENFOQUES_SUBMIT),
    'cointegracion_alpha': PARAM['cointegracion_alpha'],
    'min_meses_cointegracion': PARAM['min_meses_cointegracion'],
    'n_productos_cointegrados': int(cointegracion["cointegrado"].sum()),
    'n_productos_testeados': int(cointegracion.height),
    'archivo_productos_magicos': PARAM['archivo_productos_magicos'],
    'n_trials': {k: len(v.trials) for k, v in estudios.items()},
    'hiper_share': estudios['topdown'].best_params,
    'hiper_bottomup': estudios['bottomup'].best_params,
    'n_features': len(FEATURES), 'features': FEATURES, 'cat_features': CAT_FEATURES,
    'n_filas_train': int(tr.height),
    'n_productos_sin_prediccion': {enf: submits[enf]['sin_pred'] for enf in ENFOQUES_SUBMIT},
    'tn_total_entregado': {enf: float(submits[enf]['df']['tn'].sum()) for enf in ENFOQUES_SUBMIT},
    'semilla': PARAM['semilla'],
}
with open(DIR_OUT / "resultado.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)

fila = {
    'experimento': EXPERIMENTO, 'nivel_agregado': NIVEL,
    'densificar': PARAM['densificar'], 'max_lags': PARAM['max_lags'],
    'modelo_total': METODO_TOTAL,
    'wape_test_topdown': round(METRICAS['topdown']['test'], 5),
    'wape_test_topdown_cointegrado': round(METRICAS['topdown_cointegrado']['test'], 5),
    'wape_test_bottomup': round(METRICAS['bottomup']['test'], 5),
    'wape_test_ma3': round(METRICAS['ma3']['test'], 5),
    'wape_test_naive': round(METRICAS['naive']['test'], 5),
    'wape_val_topdown': round(METRICAS['topdown']['val'], 5),
    'brecha_test_val': round(_brecha, 5),
    'n_cointegrados': int(cointegracion["cointegrado"].sum()),
    'sin_prediccion_topdown': submits['topdown']['sin_pred'],
    'tn_entregado_topdown': round(float(submits['topdown']['df']['tn'].sum()), 1),
}
path_lb = RUTA_EXP / "leaderboard_demanda.csv"
nueva = pl.DataFrame([fila])
if path_lb.exists():
    viejo = pl.read_csv(path_lb).filter(pl.col("experimento") != EXPERIMENTO)
    nueva = pl.concat([viejo, nueva], how="diagonal_relaxed")
nueva.sort("wape_test_topdown").write_csv(path_lb)

print(f"resultado.json y leaderboard en {DIR_OUT.relative_to(BUCKET)}")
for p in sorted(DIR_OUT.iterdir()):
    print(f"  - {p.name}")
print(f"\nleaderboard_demanda.csv ({nueva.height} experimentos):")
print(nueva.select("nivel_agregado", "modelo_total", "wape_test_topdown",
                   "wape_test_topdown_cointegrado", "wape_test_bottomup", "n_cointegrados"))
